# Atlas Cartográfico e Análise Multitemporal de Cobertura da Terra
## REVIS Tabuleiro do Embaubal e RDS Vitória de Souzel (Baixo Rio Xingu, Pará)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/cartografia-embaubal-souzel/blob/main/notebooks/analise_multitemporal_mapbiomas_embaubal_souzel.ipynb)

**Instituição:** Instituto de Desenvolvimento Florestal e da Biodiversidade do Pará (IDEFLOR-Bio)  
**Diretoria:** Diretoria de Gestão e Monitoramento de Unidades de Conservação (DGMUC)  
**Execução:** Núcleo de Geotecnologias (NGEO)  
**Projeto GEE:** `ee-samuelsantosambientalcourse`  
**Fonte de Dados:** MapBiomas Brasil (Coleção 11, Série Histórica 1985–2024) via Google Earth Engine (GEE)

---

### 1. Instalação e Importação de Bibliotecas

In [ ]:
!pip install geemap earthengine-api plotly pandas --quiet

import ee
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

print("Bibliotecas importadas com sucesso!")

### 2. Autenticação e Inicialização do Google Earth Engine
> Utiliza o ID do projeto Google Cloud do usuário (`ee-samuelsantosambientalcourse`).

In [ ]:
PROJECT_ID = 'ee-samuelsantosambientalcourse'

try:
    ee.Initialize(project=PROJECT_ID)
    print(f"Earth Engine inicializado com sucesso no projeto: '{PROJECT_ID}'!")
except Exception as e:
    print(f"Solicitando autenticação para o projeto '{PROJECT_ID}'...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print(f"Earth Engine autenticado e inicializado com sucesso no projeto: '{PROJECT_ID}'!")

### 3. Carregamento dos Assets Oficiais (UCs e MapBiomas Coleção 11)

In [ ]:
# 1. Asset vetorial das Unidades de Conservação (REVIS e RDS)
asset_ucs = 'projects/ee-samuelsantosambientalcourse/assets/00_ucs_alvo_embaubal_souzel'
ucs = ee.FeatureCollection(asset_ucs)

# 2. Asset oficial MapBiomas Coleção 11 (1985 - 2024)
asset_mapbiomas = 'projects/mapbiomas-public/assets/brazil/lulc/collection11/mapbiomas_brazil_collection11_coverage_v3'
mapbiomas = ee.Image(asset_mapbiomas)

print("Assets carregados com sucesso:")
print(f" - UCs: {ucs.size().getInfo()} feições")
print(f" - MapBiomas: {len(mapbiomas.bandNames().getInfo())} bandas disponíveis")

### 4. Visualização Espacial Interativa no Mapa
Visualização das bandas de 1985 e 2024 recortadas para a área das UCs com a paleta oficial da Coleção 11.

In [ ]:
import geemap

# Tabela oficial de cores MapBiomas C11
class_ids = [1, 3, 4, 5, 6, 7, 9, 10, 11, 12, 14, 15, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 29, 30, 31, 32, 33, 35, 36, 39, 40, 41, 46, 47, 48, 49, 50, 62, 75, 77, 84, 91]
class_colors = [
    '#1f8d49', '#1f8d49', '#7dc975', '#04381d', '#007785', '#228c70', '#7a5900', '#d6bc74',
    '#519799', '#d6bc74', '#ffefc3', '#edde8e', '#E974ED', '#C27BA0', '#db7093', '#ffefc3',
    '#d4271e', '#ffa07a', '#d4271e', '#db4d4f', '#2532e4', '#ffffff', '#ad5100', '#9c0027',
    '#091077', '#fc8114', '#2532e4', '#9065d0', '#d082de', '#f5b3c8', '#c71585', '#f54ca9',
    '#d68fe2', '#9932cc', '#e6ccff', '#02d659', '#ffaa5f', '#ff69b4', '#757272', '#86b074',
    '#81dbbf', '#403d3e'
]

indices = list(range(len(class_ids)))

def style_band(band_name):
    return mapbiomas.select(band_name).clip(ucs.geometry().buffer(3000)).remap(class_ids, indices).visualize(
        min=0, max=len(indices)-1, palette=class_colors
    )

Map = geemap.Map(center=[-2.77, -51.97], zoom=11)
Map.add_basemap("HYBRID")
Map.addLayer(style_band('classification_1985'), {}, 'MapBiomas 1985', False)
Map.addLayer(style_band('classification_2024'), {}, 'MapBiomas 2024 (Recente)', True)
Map.addLayer(ucs.style(color='ffff00', fillColor='00000000', width=2), {}, 'Limites REVIS e RDS')
Map

### 5. Análise Quantitativa e Estatísticas Zonais (1985 – 2024)
Carregamento dos dados estatísticos consolidados das UCs.

In [ ]:
# URL da tabela consolidada no repositório GitHub oficial
csv_url = 'https://raw.githubusercontent.com/NGEO-IDEFLOR-Bio/cartografia-embaubal-souzel/main/camadas/tabelas/estatisticas_mapbiomas_c11_revis_rds.csv'

try:
    df = pd.read_csv(csv_url)
except Exception:
    df = pd.read_csv('../camadas/tabelas/estatisticas_mapbiomas_c11_revis_rds.csv')

# Agrupamento para a área consolidada (REVIS + RDS somadas)
df_tot = df.groupby(['Ano', 'Classe_ID', 'Classe_Nome'], as_index=False)['Area_ha'].sum()
tot_ano = df_tot.groupby('Ano')['Area_ha'].transform('sum')
df_tot['Percentual'] = (df_tot['Area_ha'] / tot_ano) * 100.0

display(df_tot[df_tot['Ano'] == 2024].sort_values('Area_ha', ascending=False))

### 6. Gráfico Interativo: Evolução Temporal da Cobertura (1985–2024)
Gráfico de barras empilhadas 100% com a paleta oficial do MapBiomas Coleção 11.

In [ ]:
cores_map = {
    6: '#007785',   # Floresta Alagável
    33: '#2532e4',  # Rio, Lago e Oceano
    11: '#519799',  # Campo Alagado
    15: '#edde8e',  # Pastagem
    21: '#ffefc3',  # Mosaico de Usos
    3: '#1f8d49',   # Formação Florestal
    12: '#d6bc74',  # Formação Campestre
    4: '#7dc975',   # Formação Savânica
    25: '#db4d4f'   # Outras Áreas não Vegetadas
}

fig_bars = go.Figure()
anos = [1985, 1995, 2005, 2015, 2024]

# Ordem de empilhamento das classes mais representativas
classes_ordem = [33, 6, 11, 15, 21, 12, 3]

for cid in classes_ordem:
    sub = df_tot[df_tot['Classe_ID'] == cid].sort_values('Ano')
    if len(sub) == 0:
        continue
    nome = sub['Classe_Nome'].iloc[0]
    fig_bars.add_trace(go.Bar(
        x=sub['Ano'],
        y=sub['Percentual'],
        name=nome,
        marker_color=cores_map.get(cid, '#888888'),
        hovertemplate='<b>%{x}</b><br>' + nome + '<br>Área: %{customdata:,.1f} ha (%{y:.1f}%)<extra></extra>',
        customdata=sub['Area_ha']
    ))

fig_bars.update_layout(
    title='<b>Evolução do Uso e Cobertura da Terra (1985 – 2024)</b><br><sup>REVIS Tabuleiro do Embaubal e RDS Vitória de Souzel (27.023 ha)</sup>',
    barmode='stack',
    xaxis=dict(title='Ano de Referência', tickmode='array', tickvals=anos),
    yaxis=dict(title='Proporção (%)', range=[0, 100]),
    template='plotly_white',
    legend=dict(title='Classe de Cobertura', orientation='h', y=-0.2),
    width=900, height=520
)
fig_bars.show()

### 7. Diagrama de Sankey Interativo (Transições 1985 → 2024)
Visualização das transições físicas de cobertura do solo entre 1985 e 2024, evidenciando estabilidade, dinâmica fluvial e antropização.

In [ ]:
# Matriz de transição oficial calculada via cruzamento de pixels
trans_url = 'https://raw.githubusercontent.com/NGEO-IDEFLOR-Bio/cartografia-embaubal-souzel/main/camadas/tabelas/transicoes_mapbiomas_1985_2024.json'
import urllib.request

try:
    with urllib.request.urlopen(trans_url) as resp:
        transicoes = json.loads(resp.read().decode('utf-8'))
except Exception:
    with open('../camadas/tabelas/transicoes_mapbiomas_1985_2024.json', encoding='utf-8') as f:
        transicoes = json.load(f)

# Filtrar transições >= 20 ha
trans_sig = [t for t in transicoes if t['ha'] >= 20.0]

labels_85 = ['Floresta Alagável (1985)', 'Rio/Lago (1985)', 'Campo Alagado (1985)', 'Mosaico de Usos (1985)', 'Formação Florestal (1985)']
labels_24 = ['Floresta Alagável (2024)', 'Rio/Lago (2024)', 'Campo Alagado (2024)', 'Pastagem (2024)', 'Mosaico de Usos (2024)', 'Formação Campestre (2024)', 'Formação Florestal (2024)']

all_labels = labels_85 + labels_24

id_to_src = {6: 0, 33: 1, 11: 2, 21: 3, 3: 4}
id_to_dst = {6: 5, 33: 6, 11: 7, 15: 8, 21: 9, 12: 10, 3: 11}

sources = []
targets = []
values = []
colors_links = []

cores_rgba = {
    6: 'rgba(0, 119, 133, 0.45)',
    33: 'rgba(37, 50, 228, 0.45)',
    11: 'rgba(81, 151, 153, 0.45)',
    21: 'rgba(255, 239, 195, 0.45)',
    3: 'rgba(31, 141, 73, 0.45)'
}

for t in trans_sig:
    s_id = t['from']
    d_id = t['to']
    if s_id in id_to_src and d_id in id_to_dst:
        sources.append(id_to_src[s_id])
        targets.append(id_to_dst[d_id])
        values.append(t['ha'])
        colors_links.append(cores_rgba.get(s_id, 'rgba(128,128,128,0.4)'))

fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=18,
        thickness=20,
        line=dict(color='black', width=0.5),
        label=all_labels,
        color=['#007785', '#2532e4', '#519799', '#ffefc3', '#1f8d49', '#007785', '#2532e4', '#519799', '#edde8e', '#ffefc3', '#d6bc74', '#1f8d49']
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color=colors_links,
        hovertemplate='%{source.label} → %{target.label}<br>Área: %{value:,.1f} ha<extra></extra>'
    )
)])

fig_sankey.update_layout(
    title='<b>Diagrama de Sankey: Transições de Cobertura da Terra (1985 → 2024)</b><br><sup>Dinâmica de Estabilidade, Erosão/Sedimentação Fluvial e Pastagem</sup>',
    template='plotly_white',
    width=960, height=560
)
fig_sankey.show()